In [ ]:
import re
import numpy as np
import pandas as pd

from xgboost import XGBRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_squared_error

def strip_suffix(name):
    """Remove pandas duplicate suffix such as .1, .2, .3."""
    return re.sub(r"\.\d+$", "", str(name).strip())


def parse_perturbation(label):
    """
    Parse perturbation name.
    """
    label = strip_suffix(label)
    parts = label.split("+")

    if len(parts) != 2:
        raise ValueError(f"Invalid perturbation: {label}")

    g1, g2 = parts

    if g2 == "ctrl":
        return "single", g1, None

    g1, g2 = sorted([g1, g2])
    return "double", g1, g2


def split_double_keep_order(label):
    """Split a double perturbation without changing gene order."""
    g1, g2 = str(label).strip().split("+")
    return g1, g2


def single_name(gene):
    return f"{gene}+ctrl"


TRAIN_CSV = "train_set.csv"
TEST_CSV = "test_set.csv"
OUTPUT_CSV = "prediction.csv"

train_raw = pd.read_csv(TRAIN_CSV, index_col=0)

print("Raw train shape:", train_raw.shape)

# Average duplicate perturbations
# pandas automatically adds .1, .2, etc. to duplicate column names
train_raw.columns = [
    strip_suffix(col)
    for col in train_raw.columns
]

# Average all experimental replicates with the same perturbation name
train = train_raw.T.groupby(level=0, sort=False).mean().T
print("Train shape after averaging:", train.shape)

# Separate singles and doubles
singles = {}
double_cols = []

for col in train.columns:

    try:
        ptype, g1, g2 = parse_perturbation(col)

    except ValueError:
        continue

    if ptype == "single":
        singles[single_name(g1)] = col

    else:
        double_cols.append(col)


genes = train.index.tolist()

print("Singles:", len(singles))
print("Doubles:", len(double_cols))

# training data
X_rows = []
Y_rows = []
train_pairs = []

for dcol in double_cols:

    _, g1, g2 = parse_perturbation(dcol)

    s1 = single_name(g1)
    s2 = single_name(g2)

    if s1 not in singles or s2 not in singles:
        continue

    x1 = train[singles[s1]].values.astype(float)
    x2 = train[singles[s2]].values.astype(float)

    # Interaction feature
    x_inter = x1 * x2

    X_rows.append(
        np.concatenate([x1, x2, x_inter])
    )

    Y_rows.append(
        train[dcol].values.astype(float)
    )

    train_pairs.append(f"{g1}+{g2}")


X_train = np.array(X_rows)
Y_train = np.array(Y_rows)

print("X_train:", X_train.shape)
print("Y_train:", Y_train.shape)
print("Pairs used:", len(train_pairs))

# Scaled additive baseline
num = 0.0
den = 0.0
used = 0

for dcol in double_cols:

    _, g1, g2 = parse_perturbation(dcol)

    s1 = single_name(g1)
    s2 = single_name(g2)

    if s1 not in singles or s2 not in singles:
        continue

    y = train[dcol].values.astype(float)

    x = (
        train[singles[s1]].values.astype(float)
        +
        train[singles[s2]].values.astype(float)
    )

    num += np.dot(x, y)
    den += np.dot(x, x)

    used += 1


s_global = num / den if den != 0 else 1.0

print("Global additive scale:", round(s_global, 4))

# Select top variable genes
K = 200

gene_var = Y_train.var(axis=0)

top_idx = np.argsort(-gene_var)[:K]

Y_train_top = Y_train[:, top_idx]

# Train XGBoost

base_xgb = XGBRegressor(
    objective="reg:squarederror",
    n_estimators=500,
    max_depth=3,
    learning_rate=0.04,
    subsample=0.9,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    reg_alpha=1e-3,
    tree_method="hist",
    n_jobs=-1,
    random_state=42
)

model = MultiOutputRegressor(
    base_xgb,
    n_jobs=-1
)

print(f"Training XGBoost on top-{K} genes...")

model.fit(
    X_train,
    Y_train_top
)

print("Done.")

# Training RMSE
Y_pred_train = model.predict(X_train)

rmse = np.sqrt(
    mean_squared_error(
        Y_train_top,
        Y_pred_train
    )
)

print(f"Training RMSE: {rmse:.4f}")

test = pd.read_csv(
    TEST_CSV,
    header=None,
    names=["perturbation"]
)

test["perturbation"] = (
    test["perturbation"]
    .astype(str)
    .str.strip()
)

print("Test pairs:", len(test))

X_test = []
test_pairs = []
missing = []

for pert in test["perturbation"]:

    # Keep the original order from test_set
    g1, g2 = split_double_keep_order(pert)

    s1 = single_name(g1)
    s2 = single_name(g2)

    if s1 not in singles or s2 not in singles:

        missing.append(pert)
        continue

    x1 = train[singles[s1]].values.astype(float)
    x2 = train[singles[s2]].values.astype(float)

    x_inter = x1 * x2

    X_test.append(
        np.concatenate([x1, x2, x_inter])
    )

    test_pairs.append(pert)

X_test = np.array(X_test)

Y_pred_top = model.predict(X_test)

rows = []

for i, pert in enumerate(test_pairs):

    g1, g2 = split_double_keep_order(pert)

    s1 = single_name(g1)
    s2 = single_name(g2)

    # Additive prediction for all genes
    add_vec = s_global * (
        train[singles[s1]].values.astype(float)
        +
        train[singles[s2]].values.astype(float)
    )

    # Replace top-K genes with XGBoost predictions
    y_full = add_vec.copy()

    y_full[top_idx] = Y_pred_top[i]

    for j, gene in enumerate(genes):

        rows.append({
            "gene": gene,
            "perturbation": pert,
            "expression": float(y_full[j])
        })


pred = pd.DataFrame(
    rows,
    columns=[
        "gene",
        "perturbation",
        "expression"
    ]
)

# 12. Check order and save

pred_order = (
    pred["perturbation"]
    .drop_duplicates()
    .tolist()
)

test_order = (
    test["perturbation"]
    .tolist()
)

if not missing:
    assert pred_order == test_order


pred.to_csv(
    OUTPUT_CSV,
    index=False
)

print("Saved:", OUTPUT_CSV)
print("Prediction shape:", pred.shape)

if missing:
    print("Missing test pairs:", missing)